# TRAVI — GOOGLE COLAB ON-DEMAND GPU INFERENCE BACKEND
### Qwen2-7B-Instruct with Ollama, Vietnam Tourism RAG & Adaptive Personalization

This notebook establishes an on-demand, free GPU inference server for the **Travi Vietnam Travel Platform**.
- **Hardware Target:** Google Colab T4 GPU (15GB GDDR6 VRAM)
- **Model:** `qwen2:7b` (Qwen2-7B-Instruct Q4_0 quantized GGUF, ~4.5 GB VRAM)
- **Inference Engine:** Ollama daemon with persistent VRAM pinning (`keep_alive: -1`)
- **Backend:** FastAPI with Token Guard, Semaphore Concurrency Limiter (2), and Adaptive Customer Profiling
- **Tunnel:** Cloudflare Tunnel (`pycloudflared`) exposing a temporary authenticated HTTPS endpoint.

> [!IMPORTANT]
> **Temporary Server:** This inference endpoint remains active while this Colab session is running. When the Colab runtime disconnects or restarts, the tunnel expires, and Travi seamlessly continues in Verified Deterministic Expert mode.

In [ ]:
# STEP 1: VERIFY HARDWARE & GPU AVAILABILITY
# Rule: Verify actual CUDA GPU availability before loading the model. Do not assume a GPU was assigned.
import torch
import subprocess
import sys

print("=== CHECKING GOOGLE COLAB CUDA GPU RUNTIME ===")
if not torch.cuda.is_available():
    raise SystemError("FATAL: No CUDA GPU detected. Please go to Colab Menu -> Runtime -> Change runtime type -> Select T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_mb = int(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024))
print(f"SUCCESS: Detected {gpu_name} with {vram_mb} MB VRAM.")
subprocess.run(["nvidia-smi"])


In [ ]:
# STEP 2: INSTALL DEPENDENCIES & OLLAMA SERVER
!pip install -q fastapi uvicorn httpx pydantic pycloudflared nest-asyncio kagglehub

# Install official Ollama binary on Linux
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
# STEP 3: START OLLAMA DAEMON IN BACKGROUND
# Note: Ollama binds strictly to 127.0.0.1:11434 and is NEVER exposed directly
import subprocess
import time
import httpx

print("Starting Ollama daemon on 127.0.0.1:11434...")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for Ollama to become ready
for attempt in range(20):
    try:
        resp = httpx.get("http://127.0.0.1:11434/api/tags", timeout=2.0)
        if resp.status_code == 200:
            print("Ollama daemon is healthy and ready.")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start within 20 seconds.")


In [ ]:
# STEP 4: PULL QWEN2-7B-INSTRUCT & PIN TO VRAM
print("Pulling qwen2:7b (Qwen2-7B-Instruct Q4_0 quantization, ~4.5GB VRAM)...")
subprocess.run(["ollama", "pull", "qwen2:7b"], check=True)
print("Model pulled successfully.")

# Pre-load model into GPU VRAM with infinite keep_alive
warmup_req = {
    "model": "qwen2:7b",
    "prompt": "Xin chào, sẵn sàng phục vụ du lịch Việt Nam.",
    "keep_alive": -1,
    "stream": False
}
resp = httpx.post("http://127.0.0.1:11434/api/generate", json=warmup_req, timeout=60.0)
print("Model loaded into VRAM. Initial response:", resp.json().get("response", "").strip())
!nvidia-smi


In [ ]:
# STEP 5: WRITE FASTAPI BACKEND APPLICATION SCRIPT (app.py)
# Generates app.py with RAG Grounding, Adaptive Customer Profiling, and Token Security
with open("app.py", "w", encoding="utf-8") as f:
    f.write('''import os
import sys
import json
import time
import asyncio
import logging
from typing import List, Optional, Dict, Any
from contextlib import asynccontextmanager

from fastapi import FastAPI, Header, HTTPException, Depends, Request
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import httpx

import secrets
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("travi-colab-gpu")

COLAB_AI_API_TOKEN = os.getenv("COLAB_AI_API_TOKEN") or os.getenv("TRAVI_COLAB_API_KEY")
if not COLAB_AI_API_TOKEN:
    COLAB_AI_API_TOKEN = secrets.token_urlsafe(24)
    logger.info("Generated cryptographically secure random session token.")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen2:7b")
MAX_CONCURRENT_REQUESTS = int(os.getenv("MAX_CONCURRENT_REQUESTS", "2"))
REQUEST_TIMEOUT_SECONDS = float(os.getenv("REQUEST_TIMEOUT_SECONDS", "60.0"))

inference_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

gpu_info: Dict[str, Any] = {"detected": False, "name": "Unknown", "total_memory_mb": 0, "cuda_available": False}

def verify_gpu_environment():
    global gpu_info
    try:
        import torch
        if torch.cuda.is_available():
            gpu_info = {
                "detected": True,
                "cuda_available": True,
                "device_count": torch.cuda.device_count(),
                "name": torch.cuda.get_device_name(0),
                "total_memory_mb": int(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024))
            }
        else:
            gpu_info = {"detected": False, "cuda_available": False, "name": "No CUDA device found", "total_memory_mb": 0}
    except Exception as e:
        gpu_info["error"] = str(e)

rag_corpus: List[Dict[str, Any]] = [
    {
        "id": "son-la-moc-chau",
        "name": "Khu du lịch Quốc gia Mộc Châu",
        "province": "Sơn La",
        "subregion": "Tây Bắc",
        "category": "Khu du lịch quốc gia",
        "description": "Cao nguyên Mộc Châu nổi tiếng với đồi chè trái tim, thung lũng mận Nà Ka, rừng thông bản Áng và thác Dải Yếm. Khí hậu mát mẻ quanh năm.",
        "experiences": ["Ngắm hoa mận", "Hái chè", "Uống sữa tươi Mộc Châu", "Khám phá bản làng Thái"],
        "foods": ["Bê chao Mộc Châu", "Cá suối nướng Pa Pỉnh Tộp", "Thịt trâu gác bếp", "Xôi ngũ sắc"],
        "source": "Sở Văn hóa Thể thao và Du lịch tỉnh Sơn La",
        "coordinate_status": "unverified_field_notes"
    },
    {
        "id": "son-la-hang-a-phu",
        "name": "Điểm du lịch Hang vợ chồng A Phủ",
        "province": "Sơn La",
        "subregion": "Tây Bắc",
        "category": "Điểm du lịch",
        "description": "Nằm tại bản Hồng Ngài, xã Bắc Yên. Hang A Phủ có cảnh quan địa chất độc đáo, gắn liền với tác phẩm văn học Vợ chồng A Phủ của Tô Hoài.",
        "experiences": ["Tham quan hang động", "Tìm hiểu văn hóa dân tộc H'Mông"],
        "foods": ["Thịt trâu sấy", "Rượu ngô Bắc Yên", "Gà đen bản địa"],
        "source": "Sở Văn hóa Thể thao và Du lịch tỉnh Sơn La",
        "coordinate_status": "unverified_field_notes"
    },
    {
        "id": "cultural-trong-dong-ngoc-lu",
        "name": "Trống đồng Ngọc Lũ",
        "province": "Hà Nội (Lưu giữ) / Hà Nam (Phát hiện)",
        "subregion": "Đồng bằng sông Hồng",
        "category": "Bảo vật quốc gia (Hiện vật văn hóa)",
        "description": "Bảo vật quốc gia số 1 của Việt Nam thuộc Văn hóa Đông Sơn, niên đại thế kỷ II-III TCN. Lưu giữ tại Bảo tàng Lịch sử Quốc gia (Số 1 Tràng Tiền, Hà Nội).",
        "experiences": ["Tham quan gian trưng bày Kim khí thời Hùng Vương"],
        "foods": ["Ẩm thực phố cổ Hà Nội"],
        "source": "Bảo tàng Lịch sử Quốc gia & Cục Di sản Văn hóa",
        "coordinate_status": "museum_venue_verified"
    },
    {
        "id": "cultural-xoe-thai",
        "name": "Nghệ thuật Xòe Thái",
        "province": "Sơn La, Yên Bái, Điện Biên, Lai Châu",
        "subregion": "Tây Bắc",
        "category": "Di sản văn hóa phi vật thể đại diện nhân loại (UNESCO)",
        "description": "Điệu múa dân gian truyền thống biểu trưng cho tinh thần cộng đồng, lòng hiếu khách của người Thái Tây Bắc.",
        "experiences": ["Tham gia vòng xòe hoa tại Bản Áng hoặc Bản Vặt"],
        "foods": ["Cơm lam", "Cá nướng Pa Pỉnh Tộp"],
        "source": "UNESCO & Sở VHTT&DL các tỉnh Tây Bắc",
        "coordinate_status": "intangible_cultural_heritage"
    }
]

def retrieve_rag_context(query: str, top_k: int = 3) -> List[Dict[str, Any]]:
    q_lower = query.lower()
    scored = []
    for item in rag_corpus:
        score = 0
        searchable = f"{item['name']} {item['description']} {' '.join(item['experiences'])} {' '.join(item['foods'])} {item['province']}".lower()
        for token in q_lower.split():
            if len(token) > 2 and token in searchable:
                score += 1
        if item['name'].lower() in q_lower:
            score += 5
        if score > 0:
            scored.append((score, item))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [item for _, item in scored[:top_k]]

@asynccontextmanager
async def lifespan(app: FastAPI):
    verify_gpu_environment()
    async with httpx.AsyncClient(timeout=15.0) as client:
        try:
            await client.post(f"{OLLAMA_BASE_URL}/api/generate", json={"model": MODEL_NAME, "prompt": "Xin chào", "keep_alive": -1})
        except Exception:
            pass
    yield

app = FastAPI(title="Travi Colab GPU", lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

def verify_token(authorization: Optional[str] = Header(None), x_api_key: Optional[str] = Header(None)):
    token = authorization.replace("Bearer ", "").strip() if authorization and authorization.startswith("Bearer ") else x_api_key
    if not token or token != COLAB_AI_API_TOKEN:
        raise HTTPException(status_code=401, detail="Unauthorized: Invalid API token.")
    return token

class CustomerProfile(BaseModel):
    customer_type: Optional[str] = "DOMESTIC"
    origin_country: Optional[str] = "VN"
    origin_province: Optional[str] = None
    destination: Optional[str] = None
    preferred_language: Optional[str] = "vi"
    budget: Optional[str] = "moderate"
    travel_duration: Optional[int] = 3
    interests: Optional[List[str]] = []

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    prompt: str = Field(..., min_length=1, max_length=1000)
    profile: Optional[CustomerProfile] = None
    context: Optional[Dict[str, Any]] = None
    history: Optional[List[ChatMessage]] = []
    stream: Optional[bool] = False

@app.get("/health")
async def health_check():
    ollama_ok = False
    try:
        async with httpx.AsyncClient(timeout=3.0) as client:
            r = await client.get(f"{OLLAMA_BASE_URL}/api/tags")
            ollama_ok = r.status_code == 200
    except Exception:
        pass
    return {
        "status": "healthy" if ollama_ok else "offline",
        "service": "travi-colab-gpu-inference",
        "gpu": gpu_info,
        "model": MODEL_NAME,
        "concurrency": {"max": MAX_CONCURRENT_REQUESTS, "available": inference_semaphore._value}
    }

@app.get("/api/model-info")
async def model_info(token: str = Depends(verify_token)):
    return {
        "model": MODEL_NAME,
        "quantization": "Q4_0 (GGUF 4-bit Quantization)",
        "gpu": gpu_info,
        "rag_facts_count": len(rag_corpus),
        "keep_alive": "infinite (-1)",
        "runtime_expires_note": "Temporary Google Colab runtime. Expires when notebook disconnects."
    }

@app.post("/api/chat")
async def chat(request: ChatRequest, token: str = Depends(verify_token)):
    retrieved = retrieve_rag_context(request.prompt, top_k=2)
    rag_text = "\n".join([f"- {f['name']} ({f['province']}): {f['description']} | Đặc sản: {', '.join(f.get('foods', []))}" for f in retrieved]) if retrieved else ""
    cust_type = (request.profile.customer_type if request.profile and request.profile.customer_type else "DOMESTIC").upper()

    prompt_guide = ""
    if cust_type == "INTERNATIONAL":
        prompt_guide = "Khách quốc tế: Hướng dẫn văn hóa, song ngữ, giải thích nghi thức đền chùa và thủ tục thực tế."
    elif cust_type == "LOCAL":
        prompt_guide = "Khách bản địa: Ưu tiên điểm đến ngách (hidden gems), ẩm thực ít người biết, tránh tụ điểm đông đúc."
    elif cust_type == "INTERPROVINCIAL":
        prompt_guide = "Khách liên tỉnh: Tối ưu cung đường đèo dốc, thời gian di chuyển và kết nối giữa các tỉnh lân cận."
    else:
        prompt_guide = "Khách nội địa: Lịch trình cân đối chi phí, ẩm thực vùng miền chuẩn vị và danh lam thắng cảnh nổi tiếng."

    system_prompt = f"Bạn là Travi AI — Trợ lý chuyên gia du lịch Việt Nam tận tâm.\nNguyên tắc: Tuyệt đối không bịa đặt giá vé hay tọa độ. Sử dụng sự thật xác thực.\nĐịnh hướng: {prompt_guide}"
    if rag_text:
        system_prompt += f"\n\nRetrieved Facts:\n{rag_text}"

    messages = [{"role": "system", "content": system_prompt}]
    for h in (request.history or [])[-4:]:
        messages.append({"role": h.role, "content": h.content})
    messages.append({"role": "user", "content": request.prompt})

    async with inference_semaphore:
        async with httpx.AsyncClient(timeout=REQUEST_TIMEOUT_SECONDS) as client:
            payload = {"model": MODEL_NAME, "messages": messages, "stream": False, "keep_alive": -1}
            resp = await client.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload)
            if resp.status_code != 200:
                raise HTTPException(status_code=502, detail=resp.text)
            reply = resp.json().get("message", {}).get("content", "")
            return {
                "status": "success",
                "content": reply,
                "sources": [{"name": f["name"], "province": f["province"], "source": f["source"]} for f in retrieved],
                "personalized_for": cust_type,
                "provenance": {
                    "providerName": f"Google Colab GPU Qwen2-7B ({gpu_info.get('name', 'T4')})",
                    "isLiveAi": True,
                    "modeDescription": "Ollama Serverless GPU Container via Secure HTTPS Tunnel",
                    "vramAllocatedMB": gpu_info.get("total_memory_mb", 15360)
                }
            }
''')
print("Saved app.py with RAG Grounding, Adaptive Profiles, and Token Security.")


In [ ]:
# STEP 6: START FASTAPI & LAUNCH SECURE CLOUDFLARE TUNNEL
import threading
import uvicorn
from pycloudflared import try_cloudflare
import app as colab_app

def run_server():
    uvicorn.run("app:app", host="127.0.0.1", port=8000, log_level="warning")

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(3)

# Launch temporary Cloudflare HTTPS tunnel to expose 127.0.0.1:8000
tunnel = try_cloudflare(port=8000)
public_url = tunnel.tunnel.strip()
session_token = colab_app.COLAB_AI_API_TOKEN

print("=" * 75)
print("🚀 TRAVI GOOGLE COLAB GPU INFERENCE BACKEND IS ONLINE!")
print(f"Public HTTPS Endpoint: {public_url}")
print(f"API Authentication Token: {session_token}")
print(f"Health Check URL: {public_url}/health")
print(f"Chat Endpoint: {public_url}/api/chat")
print("=" * 75)
print("COPY THESE ENVIRONMENT VARIABLES INTO YOUR TRAVI SERVER (.env):")
print(f"COLAB_AI_BASE_URL={public_url}")
print(f"COLAB_AI_API_TOKEN={session_token}")
print("COLAB_AI_ENABLED=true")
print("=" * 75)


In [ ]:
# STEP 7: AUTOMATED VERIFICATION SELF-TESTS
import httpx
headers = {"Authorization": f"Bearer {colab_app.COLAB_AI_API_TOKEN}"}
client = httpx.Client(base_url="http://127.0.0.1:8000", headers=headers, timeout=30.0)

# 1. Health check
h = client.get("/health").json()
print("1. Health Check:", h["status"], "GPU:", h["gpu"]["name"])
assert h["status"] == "healthy"

# 2. Auth guard
unauth = httpx.get("http://127.0.0.1:8000/api/model-info")
print("2. Auth Guard Status:", unauth.status_code)
assert unauth.status_code == 401

# 3. Model Info with auth
info = client.get("/api/model-info").json()
print("3. Model Info:", info["model"], "Quantization:", info["quantization"])

# 4. Adaptive Chat - International Profile
intl_req = {
    "prompt": "I am traveling to Hanoi. What temple etiquette should I follow?",
    "profile": {"customer_type": "INTERNATIONAL", "origin_country": "France", "destination": "Hà Nội"}
}
ans_intl = client.post("/api/chat", json=intl_req).json()
print("4. International Profile Reply Preview:", ans_intl["content"][:150] + "...")
assert ans_intl["personalized_for"] == "INTERNATIONAL"

# 5. Adaptive Chat - Local Profile
local_req = {
    "prompt": "Có quán ăn nào yên tĩnh ở Mộc Châu không?",
    "profile": {"customer_type": "LOCAL", "origin_province": "Sơn La", "destination": "Mộc Châu"}
}
ans_local = client.post("/api/chat", json=local_req).json()
print("5. Local Profile Reply Preview:", ans_local["content"][:150] + "...")
assert ans_local["personalized_for"] == "LOCAL"

print("\nALL COLAB BACKEND TESTS PASSED!")
